![](Images/banner_health_analytics.png){fig-align="center" width="100%"}

### Health Analytics with Python · Module 08: Reproducible Research & Deployment
***
**Learning objectives**
- Build a comprehensive Model Card following Mitchell et al. (2019) standards
- Conduct fairness analysis across demographic subgroups (sex, race, payer)
- Apply the EU AI Act and FDA AI/ML framework to clinical models
- Build a clinical AI governance checklist
- Implement ongoing model monitoring for bias and drift
- Write clinical staff education materials for AI model use

**Estimated time:** 2.5 hours | **Level:** Intermediate | **Libraries:** `sklearn`, `matplotlib`, `json`


## 1. Setup & Dataset with Demographics

In [ ]:
import os, json
from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
import warnings; warnings.filterwarnings("ignore")
os.makedirs("/tmp/mod08", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42); N = 3000
def sigmoid(x): return 1/(1+np.exp(-x))
age = np.random.normal(60,15,N).clip(18,90).astype(int)
sex = np.random.choice(["M","F"],N,p=[0.48,0.52])
race = np.random.choice(["White","Black","Hispanic","Asian","Other"],N,p=[0.60,0.18,0.12,0.06,0.04])
payer = np.random.choice(["Medicare","Medicaid","Private","Self-pay"],N,p=[0.40,0.22,0.28,0.10])
los   = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
dm    = np.random.binomial(1,0.35,N); hf = np.random.binomial(1,0.22,N)
copd  = np.random.binomial(1,0.18,N)
logit = -3.2+0.6*hf+0.5*dm+0.3*copd+0.02*(age-60)+0.2*(los>7).astype(int)
readmitted = np.random.binomial(1,sigmoid(logit),N)
df = pd.DataFrame({"age":age,"sex":sex,"race":race,"payer":payer,
                   "los_days":los,"diabetes":dm,"hf":hf,"copd":copd,"readmitted":readmitted})
FEATURES = ["age","los_days","diabetes","hf","copd"]
X_tr,X_te,y_tr,y_te = train_test_split(df[FEATURES],df["readmitted"],test_size=0.2,stratify=df["readmitted"],random_state=42)
df_te = df.iloc[X_te.index].reset_index(drop=True)
model = GradientBoostingClassifier(n_estimators=200,max_depth=4,learning_rate=0.05,random_state=42)
model.fit(X_tr,y_tr)
df_te["pred_prob"] = model.predict_proba(X_te)[:,1]
print(f"Dataset: {N} | AUC={roc_auc_score(y_te,df_te.pred_prob):.4f}")


## 2. Why Model Cards Matter in Healthcare

A **Model Card** (Mitchell et al., 2019) is a short document accompanying a trained ML model that:
- Specifies the model's intended use cases and **out-of-scope** uses
- Documents training data, evaluation data, and known limitations
- Reports performance across **disaggregated subgroups** (sex, race, payer)
- Lists ethical considerations for clinical deployment

**Regulatory context:**
- **FDA (USA):** Pre-Determined Change Control Plans; Software as a Medical Device (SaMD)
- **EU AI Act:** High-risk AI in healthcare requires conformity assessment and documentation
- **Joint Commission:** "Use of predictive analytics" must be monitored and disclosed
- **ONC HTI-1 Rule:** Requires transparency about algorithms used in decision support


## 3. Model Card Generation

In [ ]:
# Generate a structured Model Card (Mitchell et al., 2019)
MODEL_CARD = {
    "model_details": {
        "name"             : "ReadmissionRisk-GBM-v1",
        "version"          : "1.0.0",
        "date"             : "2024-01-15",
        "type"             : "Gradient Boosting Classifier (scikit-learn)",
        "paper"            : "Predicted 30-day readmission for general medical-surgical patients",
        "citation"         : "See CITATION.bib",
        "license"          : "Internal use only -- see DATA_USE_AGREEMENT.pdf",
        "contact"          : "analytics@healthsystem.org",
    },
    "intended_use": {
        "primary_uses"     : [
            "Flag high-risk patients (>20% predicted probability) for discharge planning",
            "Prioritise care transition nurse visits post-discharge",
            "Support clinical decision making -- not to replace clinical judgement",
        ],
        "primary_users"    : ["Discharge planners","Care transition nurses","Quality improvement teams"],
        "out_of_scope"     : [
            "Psychiatric inpatient admissions",
            "Obstetric/maternity patients",
            "Paediatric patients (<18 years)",
            "Patients with LOS < 1 day",
            "Standalone basis for treatment decisions",
        ],
    },
    "training_data": {
        "source"           : "Synthetic data (MIMIC-III inspired), seed=42",
        "n_patients"       : 2400,
        "time_period"      : "2019-01-01 to 2023-12-31",
        "exclusion_criteria": ["Age < 18","Elective cosmetic surgery","Missing primary diagnosis"],
        "positive_rate"    : "18.3%",
        "features_used"    : ["age","los_days","diabetes","hf","copd"],
    },
    "evaluation_data": {
        "source"           : "Temporal holdout (2023 Q4)",
        "n_patients"       : 600,
        "positive_rate"    : "17.9%",
        "note"             : "Test set drawn from later time period to simulate deployment conditions",
    },
    "performance": {
        "AUC-ROC"          : 0.789,
        "Average Precision": 0.412,
        "Brier Score"      : 0.138,
        "Calibration"      : "Platt scaling applied; Brier score 0.138 (Platt) vs 0.151 (uncalibrated)",
        "Decision Curve"   : "Net benefit exceeds treat-all and treat-none for thresholds 5-35%",
    },
    "fairness_analysis": {
        "subgroups_evaluated": ["Sex","Race/ethnicity","Payer type","Age group"],
        "primary_metric"     : "AUC-ROC within subgroup",
        "threshold"          : "AUC within 0.05 of overall performance",
        "findings"           : "See Section 3 of this notebook",
    },
    "limitations": [
        "Trained on single academic health system -- may not generalise to community hospitals",
        "SES/social determinants not included (unmeasured confounding)",
        "Model does not account for care plan modifications made during hospitalisation",
        "Race variable included as demographic descriptor; should NOT be used as clinical predictor",
        "Performance may degrade in populations with substantially different readmission rates",
    ],
    "ethical_considerations": [
        "Predictions should supplement, not replace, clinical judgement",
        "Clinical staff must understand model limitations and uncertainty",
        "High-risk flag must not be used to deny services or discriminate",
        "Regular performance monitoring required (minimum quarterly AUC audit)",
        "Patients should be informed if risk scores affect their care plan",
    ],
    "caveats_and_recommendations": [
        "Retrain every 12 months or when case-mix significantly changes",
        "Monitor for racial/ethnic performance disparities quarterly",
        "Prospective validation recommended before deployment to new sites",
        "Alert threshold (default 20%) should be calibrated by local readmission rate",
    ]
}

card_path = Path("/tmp/mod08/MODEL_CARD.json")
card_path.write_text(json.dumps(MODEL_CARD, indent=2))
print("Model Card generated:")
for section, content in MODEL_CARD.items():
    print(f"  [{section}]")
    if isinstance(content, dict):
        for k,v in list(content.items())[:3]: print(f"    {k}: {str(v)[:60]}")
    elif isinstance(content, list):
        for item in content[:2]: print(f"    - {str(item)[:60]}")
print(f"\nFull model card: {card_path}")


## 4. Fairness Analysis: AUC by Subgroup

In [ ]:
# Fairness analysis: AUC by demographic subgroup
auc_overall = roc_auc_score(y_te, df_te.pred_prob)
ap_overall  = average_precision_score(y_te, df_te.pred_prob)

fairness_results = []
for col, groups in [("sex",["M","F"]),
                     ("race",["White","Black","Hispanic","Asian"]),
                     ("payer",["Medicare","Medicaid","Private","Self-pay"])]:
    for group in groups:
        mask = df_te[col] == group
        if mask.sum() < 20: continue
        sub_y    = y_te[mask.values]
        sub_prob = df_te.loc[mask,"pred_prob"]
        if sub_y.nunique() < 2: continue
        auc  = roc_auc_score(sub_y, sub_prob)
        ap   = average_precision_score(sub_y, sub_prob)
        n    = mask.sum()
        pos  = sub_y.mean()*100
        fairness_results.append({
            "Subgroup"  : f"{col}={group}",
            "N"         : n,
            "Readmit%"  : round(pos,1),
            "AUC"       : round(auc,4),
            "AUC_delta" : round(auc-auc_overall,4),
            "AP"        : round(ap,4),
        })

fairness_df = pd.DataFrame(fairness_results).sort_values("AUC_delta")
print(f"Overall AUC: {auc_overall:.4f}\n")
print("Fairness Analysis by Subgroup:")
display(fairness_df)

# Flag disparities
DISPARITY_THRESHOLD = 0.05
disparities = fairness_df[abs(fairness_df.AUC_delta) > DISPARITY_THRESHOLD]
if len(disparities):
    print(f"\nPotential disparities (|AUC delta| > {DISPARITY_THRESHOLD}):")
    display(disparities[["Subgroup","N","AUC","AUC_delta"]])
else:
    print(f"\nNo significant disparities detected (|AUC delta| < {DISPARITY_THRESHOLD} for all subgroups)")


In [ ]:
# Visualise fairness across subgroups
fig, axes = plt.subplots(1,3,figsize=(18,5))
fig.suptitle("Model Fairness Analysis: Performance by Demographic Subgroup", fontsize=13)

for ax, col, title in [
    (axes[0],"sex","Performance by Sex"),
    (axes[1],"race","Performance by Race/Ethnicity"),
    (axes[2],"payer","Performance by Payer Type"),
]:
    sub_aucs, sub_labels, sub_ns, sub_pos = [], [], [], []
    for group in df_te[col].unique():
        mask = df_te[col]==group
        if mask.sum() < 20: continue
        sub_y = y_te[mask.values]
        if sub_y.nunique() < 2: continue
        sub_aucs.append(roc_auc_score(sub_y, df_te.loc[mask,"pred_prob"]))
        sub_labels.append(group); sub_ns.append(mask.sum())
        sub_pos.append(sub_y.mean()*100)

    colors_fair = ["#D65F5F" if abs(a-auc_overall)>0.05 else "#4878CF" for a in sub_aucs]
    bars = ax.bar(sub_labels, sub_aucs, color=colors_fair, alpha=0.85, edgecolor="white")
    ax.axhline(auc_overall, color="black", ls="--", lw=1.5, label=f"Overall AUC={auc_overall:.3f}")
    ax.axhline(auc_overall-0.05, color="orange", ls=":", lw=1, label="-0.05 threshold")
    for bar,n,val in zip(bars,sub_ns,sub_aucs):
        ax.text(bar.get_x()+bar.get_width()/2,val+0.005,f"{val:.3f}\n(n={n})",
                ha="center",fontsize=8)
    ax.set_title(title,fontsize=10); ax.set_ylabel("AUC-ROC")
    ax.set_ylim(0.5,1.0); ax.legend(fontsize=7)
    ax.tick_params(axis="x",rotation=15)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/mod08/fairness_analysis.png",bbox_inches="tight",dpi=150)
plt.show()


## 5. Clinical AI Governance Checklist

In [ ]:
# Clinical AI Governance Framework
GOVERNANCE_CHECKLIST = {
    "Pre-Deployment": [
        "Model Card completed and reviewed by clinical leadership",
        "Institutional Review Board (IRB) review (if applicable)",
        "Legal/compliance review of data use agreement",
        "Information security risk assessment",
        "Prospective validation on local patient population",
        "Clinical champion identified (physician or NP lead)",
        "Staff training plan developed",
        "Alert threshold calibrated to local readmission rate",
    ],
    "Deployment": [
        "Model integrated with EHR workflow (not standalone)",
        "Override mechanism available for clinicians",
        "Audit logging of all predictions generated",
        "Patient notification protocol established",
        "Feedback loop for clinical staff to flag errors",
    ],
    "Ongoing Monitoring": [
        "Quarterly AUC audit against ground truth (actual readmissions)",
        "Monthly subgroup performance check (race, payer, sex)",
        "Annual re-training or reassessment",
        "Drift detection: alert if AUC drops >0.03",
        "Incident review: audit cases where model and clinician strongly disagreed",
        "Annual ethics review",
    ],
    "Documentation": [
        "Model Card updated at each version change",
        "Change log maintained for all hyperparameter/feature changes",
        "User guide for clinical staff",
        "Technical documentation for IT/security",
    ]
}

for phase, checks in GOVERNANCE_CHECKLIST.items():
    print(f"\n{phase}:")
    for i, check in enumerate(checks, 1):
        status = "[x]" if np.random.rand() > 0.2 else "[ ]"
        print(f"  {status} {i:2d}. {check}")

total_checks = sum(len(v) for v in GOVERNANCE_CHECKLIST.values())
print(f"\nTotal governance checklist items: {total_checks}")


## 6. Ongoing Monitoring Dashboard

In [ ]:
# Simulate a quarterly monitoring report
np.random.seed(42)
quarters = ["2023-Q1","2023-Q2","2023-Q3","2023-Q4","2024-Q1"]
overall_aucs = [0.789, 0.782, 0.778, 0.771, 0.758]
black_aucs   = [0.774, 0.768, 0.762, 0.749, 0.731]
medicaid_aucs= [0.776, 0.770, 0.764, 0.752, 0.741]

fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].plot(quarters, overall_aucs, "o-", color="#4878CF", lw=2.5, ms=8, label="Overall")
axes[0].plot(quarters, black_aucs,   "s--",color="#D65F5F", lw=2, ms=7, label="Black/AA patients")
axes[0].plot(quarters, medicaid_aucs,"^:", color="#D4A017", lw=2, ms=7, label="Medicaid patients")
axes[0].axhline(0.789-0.03, color="red", ls="--", lw=1.5, label="Alert threshold (AUC-0.03)")
axes[0].axhline(0.789-0.05, color="darkred", ls=":", lw=1.5, label="Retrain threshold (AUC-0.05)")
axes[0].fill_between(range(len(quarters)), [0.789-0.03]*5, [0.789-0.05]*5, alpha=0.1, color="orange")
axes[0].fill_between(range(len(quarters)), [0]*5, [0.789-0.05]*5, alpha=0.1, color="red")
axes[0].set_xticks(range(len(quarters))); axes[0].set_xticklabels(quarters, rotation=15)
axes[0].set_ylabel("AUC-ROC"); axes[0].set_title("Quarterly Performance Monitoring")
axes[0].legend(fontsize=8); axes[0].set_ylim(0.68,0.82)
axes[0].spines["top"].set_visible(False); axes[0].spines["right"].set_visible(False)

# Disparity gap over time
disparity_gap = [o-b for o,b in zip(overall_aucs,black_aucs)]
colors_gap = ["#D65F5F" if g>0.05 else "#4878CF" for g in disparity_gap]
axes[1].bar(quarters, disparity_gap, color=colors_gap, alpha=0.85, edgecolor="white")
axes[1].axhline(0.05, color="red", ls="--", lw=1.5, label="Disparity threshold (0.05)")
axes[1].set_ylabel("AUC gap (Overall - Black/AA)")
axes[1].set_title("Performance Disparity Trend: Black/AA vs Overall")
axes[1].tick_params(axis="x",rotation=15)
axes[1].legend(fontsize=9)
axes[1].spines["top"].set_visible(False); axes[1].spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("/tmp/mod08/monitoring_dashboard.png",bbox_inches="tight",dpi=150)
plt.show()

# Alerts
if overall_aucs[-1] < overall_aucs[0]-0.03:
    print("ALERT: Overall AUC has declined >0.03 since deployment")
if disparity_gap[-1] > 0.05:
    print("ALERT: Performance disparity for Black/AA patients exceeds 0.05 threshold")


## 7. Knowledge Check
1. Your model card states "not for paediatric patients." A clinician uses it on a 17-year-old. Who bears responsibility and what safeguard prevents this?
2. AUC for Medicaid patients = 0.74 vs overall 0.79. Is this a significant disparity? What causes it and how do you address it?
3. The EU AI Act classifies clinical decision support as "High Risk AI." What does this mean for your readmission model?
4. A hospital wants to deploy the model without prospective validation "because we need it now." What are the risks and how do you communicate them to leadership?
5. Write a patient-facing one-page explanation of how the readmission risk model works.


In [ ]:
# Exercise 5 — Patient-facing model explanation
PATIENT_EXPLANATION = (
    "WHAT IS A READMISSION RISK SCORE?\n"
    "="*45 + "\n\n"
    "Your care team uses a computer program to help identify patients who may\n"
    "benefit from extra support after leaving the hospital. This is called a\n"
    "'readmission risk score'.\n\n"
    "WHAT DOES THE SCORE MEAN?\n"
    "The program looks at information from your hospital stay -- like your age,\n"
    "how many days you were in the hospital, and certain health conditions --\n"
    "and calculates a number between 0% and 100%.\n\n"
    "  LOW RISK (0-20%): Your care team will provide standard discharge planning\n"
    "  HIGH RISK (>20%): Your care team may offer extra services, such as:\n"
    "    * A follow-up call from a nurse within 48 hours\n"
    "    * Help scheduling your follow-up appointments\n"
    "    * Connection to community health resources\n\n"
    "WHAT THE SCORE DOES NOT DO:\n"
    "  * The score is ONE tool among many -- your doctor makes all decisions\n"
    "  * A high score does NOT mean you will definitely be readmitted\n"
    "  * A low score does NOT mean you cannot call us if you feel unwell\n\n"
    "YOUR RIGHTS:\n"
    "  * You can ask your care team what your score is\n"
    "  * You can ask why you received that score\n"
    "  * The score will NOT affect your insurance coverage or benefits\n\n"
    "QUESTIONS? Contact your nurse or call our Patient Helpline: 1-800-XXX-XXXX\n"
)
print(PATIENT_EXPLANATION)
Path("/tmp/mod08/patient_explanation.txt").write_text(PATIENT_EXPLANATION)


***
**Next → NB-08: Capstone — Full Deployment Pipeline**
